# ML-05 — Feature Vector and Leakage/Privacy Check

This notebook defines the 11-variable leakage-safe candidate vector. The final clustering model later selects five dimensions from this candidate set; candidate status is not the same as final-model inclusion.

## 1. Candidate feature vector

Candidate variables capture reach, click volume, session volume, page length, freshness, ranking, click efficiency, engagement, scrolling, age, and update ratio. Counts are log-transformed. Missingness is measured before any imputation so it cannot be mistaken for a real value.

In [ ]:
import os
import numpy as np
import pandas as pd

possible_paths = [
    '../../data/raw/content_refresh_anonymized.csv',
    'data/raw/content_refresh_anonymized.csv',
    '/content/content_refresh_anonymized.csv',
]
data_path = next((path for path in possible_paths if os.path.exists(path)), None)
if data_path is None:
    raise FileNotFoundError('Could not locate content_refresh_anonymized.csv')

df_raw = pd.read_csv(data_path)
df_contract = df_raw.loc[
    (df_raw['impressions_90d'] >= 10)
    & (df_raw['content_age_days'] >= 90)
] .copy()

missingness = df_contract[['word_count', 'scroll_rate']].isna().mean().rename('missing_share')
df_features = pd.DataFrame({
    'impressions_log': np.log1p(df_contract['impressions_90d']),
    'clicks_log': np.log1p(df_contract['clicks_90d']),
    'sessions_log': np.log1p(df_contract['sessions_90d']),
    'word_count_log': np.log1p(df_contract['word_count'].fillna(df_contract['word_count'].median())),
    'staleness_log': np.log1p(df_contract['days_since_last_update']),
    'avg_position': df_contract['avg_position'].replace(0, np.nan).fillna(df_contract['avg_position'].replace(0, np.nan).median()),
    'ctr': df_contract['ctr'],
    'engagement_rate': df_contract['engagement_rate'],
    'scroll_rate': df_contract['scroll_rate'].fillna(df_contract['scroll_rate'].median()),
    'content_age_days': df_contract['content_age_days'],
    'update_ratio': np.clip(
        df_contract['days_since_last_update'] / (df_contract['content_age_days'] + 1), 0, 1
    ),
})

print(f'Candidate matrix: {df_features.shape[0]:,} rows x {df_features.shape[1]} features')
display(missingness.to_frame().round(3))
assert df_features.shape[1] == 11
assert df_features.notna().all().all()


## 2. Candidate notes and final-model transition

| Candidate | Available at decision time? | Final model? | Rationale |
| --- | --- | --- | --- |
| `impressions_log` | Yes | Yes | Reach |
| `clicks_log` | Yes | No | Overlaps with impressions |
| `sessions_log` | Yes | No | Overlaps with impressions |
| `word_count_log` | Yes | No | Missingness and weak behavioral relevance |
| `staleness_log` | Yes | Yes | Freshness |
| `avg_position` | Yes | Yes | Ranking; zero is treated as missing |
| `ctr` | Yes | Yes | Click efficiency |
| `engagement_rate` | Yes | Yes | User behavior |
| `scroll_rate` | Yes | No | Additional engagement measure |
| `content_age_days` | Yes | No | Less direct than freshness |
| `update_ratio` | Yes | No | Derived and overlaps with staleness |

The five final features are therefore `impressions_log`, `avg_position`, `ctr`, `staleness_log`, and `engagement_rate`.

In [ ]:
overlap = df_features.corr().loc[
    ['impressions_log', 'staleness_log', 'engagement_rate'],
    ['clicks_log', 'sessions_log', 'update_ratio', 'scroll_rate'],
]
display(overlap.round(3))
final_features = [
    'impressions_log', 'avg_position', 'ctr', 'staleness_log', 'engagement_rate'
]
assert set(final_features).issubset(df_features.columns)


## 3. Leakage, privacy, and decision-time audit

The model must not use target-derived trend fields, product scores/actions, identifiers, URLs, or raw query text. Correlation with a downstream decline proxy is diagnostic only; the principal control is exclusion by definition before model fitting.

In [ ]:
decline_proxy = (df_contract['trend_direction'] == 'down').astype(int)
correlation_audit = df_features[final_features].corrwith(decline_proxy)
display(correlation_audit.rename('correlation_with_decline_proxy').to_frame().round(3))

forbidden_fields = {
    'trend_direction', 'trend_pct', 'is_declining_label',
    'health_score', 'priority_score', 'action_type',
    'content_id', 'client_id', 'url', 'keyword_text',
}
assert not (set(final_features) & forbidden_fields)
print('Passed: final features exclude target-derived, product-output, and identifying fields.')


## Self-check

- [x] The 11 candidate variables and five final variables are explicitly connected.
- [x] Missingness and position-zero semantics are documented.
- [x] Leakage, product circularity, privacy, and decision-time controls are explicit.